Before running the full metric, I'm going to run a severity score utilizing the HRD attacks and other allegations.

In [2]:
!pip install scikit-learn pandas openpyxl

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 16.7 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 18.2 MB/s  0:00:01 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("All good")

All good


In [4]:
import os
os.listdir()

['HRRR_metric.ipynb', '.ipynb_checkpoints']

In [8]:
df = pd.read_excel("/Users/pablopadres/Documents/School/MS2/thesis/Data/raw_data/human_rights_risk_&_responsiveness.xlsx")

df.head()

,Brand,HRD_Attacks,Other_Allegations,Response_Requests,Response_Rate
0,GAP,3,10,25,96
1,Banana Republic,3,10,25,96
2,Old Navy,0,0,2,50
3,American Eagle,0,6,7,72
4,Abercrombie & Fitch,0,0,3,67


In [10]:
severity_cols = ['HRD_Attacks', 'Other_Allegations']
X_severity = df[severity_cols].astype(float)

X_severity.head()

,HRD_Attacks,Other_Allegations
0,3.0,10.0
1,3.0,10.0
2,0.0,0.0
3,0.0,6.0
4,0.0,0.0


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_severity_scaled = scaler.fit_transform(X_severity)

X_severity_scaled

array([[ 3.49929037,  0.88203092],
       [ 3.49929037,  0.88203092],
       [-0.40689423, -0.81826965],
       [-0.40689423,  0.20191069],
       [-0.40689423, -0.81826965],
       [-0.40689423,  3.77254189],
       [-0.40689423,  2.07224132],
       [-0.40689423,  0.54197081],
       [-0.40689423,  0.37194075],
       [-0.40689423, -0.47820953],
       [ 0.8951673 ,  1.05206098],
       [-0.40689423,  0.88203092],
       [ 0.8951673 ,  0.88203092],
       [-0.40689423, -0.81826965],
       [ 0.8951673 , -0.81826965],
       [-0.40689423,  0.03188064],
       [ 0.8951673 , -0.30817948],
       [-0.40689423, -0.81826965],
       [-0.40689423,  0.37194075],
       [-0.40689423, -0.81826965],
       [-0.40689423, -0.64823959],
       [-0.40689423, -0.47820953],
       [-0.40689423, -0.47820953],
       [-0.40689423, -0.81826965],
       [-0.40689423, -0.47820953],
       [-0.40689423,  0.20191069],
       [-0.40689423, -0.81826965],
       [-0.40689423, -0.81826965],
       [-0.40689423,

In [12]:
from sklearn.decomposition import PCA

pca_severity = PCA()
severity_components = pca_severity.fit_transform(X_severity_scaled)

severity_components

array([[ 3.098062  ,  1.85068191],
       [ 3.098062  ,  1.85068191],
       [-0.86632169,  0.29088635],
       [-0.14494525, -0.43049009],
       [-0.86632169,  0.29088635],
       [ 2.37987228, -2.95530762],
       [ 1.17757822, -1.75301356],
       [ 0.09551356, -0.6709489 ],
       [-0.02471584, -0.55071949],
       [-0.62586287,  0.05042754],
       [ 1.37689832, -0.11094058],
       [ 0.33597238, -0.91140771],
       [ 1.25666892,  0.00928883],
       [-0.86632169,  0.29088635],
       [ 0.05437485,  1.21158289],
       [-0.26517466, -0.31026068],
       [ 0.41506307,  0.85089467],
       [-0.86632169,  0.29088635],
       [-0.02471584, -0.55071949],
       [-0.86632169,  0.29088635],
       [-0.74609228,  0.17065694],
       [-0.62586287,  0.05042754],
       [-0.62586287,  0.05042754],
       [-0.86632169,  0.29088635],
       [-0.62586287,  0.05042754],
       [-0.14494525, -0.43049009],
       [-0.86632169,  0.29088635],
       [-0.86632169,  0.29088635],
       [-0.62586287,

In [13]:
pca_severity.explained_variance_ratio_

array([0.62409925, 0.37590075])

In [14]:
import pandas as pd

severity_loadings = pd.DataFrame(
    pca_severity.components_.T,
    columns=['PC1', 'PC2'],
    index=severity_cols
)

severity_loadings

,PC1,PC2
HRD_Attacks,0.707107,0.707107
Other_Allegations,0.707107,-0.707107


In [15]:
severity_weights = abs(severity_loadings['PC1'])
severity_weights = severity_weights / severity_weights.sum()

severity_weights

HRD_Attacks          0.5
Other_Allegations    0.5
Name: PC1, dtype: float64

In [16]:
df['severity_score_raw'] = (
    severity_weights['HRD_Attacks'] * df['HRD_Attacks'] +
    severity_weights['Other_Allegations'] * df['Other_Allegations']
)

In [21]:
def normalize(series):
    if series.max() == series.min():
        return pd.Series([50]*len(series))  # fallback
    return 100 * (series - series.min()) / (series.max() - series.min())

In [22]:
df['severity_score'] = 100 - normalize(df['severity_score_raw'])

df[['Brand', 'severity_score']]

,Brand,severity_score
0,GAP,5.185185e+01
1,Banana Republic,5.185185e+01
2,Old Navy,1.000000e+02
3,American Eagle,7.777778e+01
4,Abercrombie & Fitch,1.000000e+02
5,Nike,-1.421085e-14
6,Under Armour,3.703704e+01
7,Lululemon,7.037037e+01
8,New Balance,7.407407e+01
9,Champion,9.259259e+01


NORMALIZED SCORES ARE NOT THE DESIRE RESULTS, SO CELLS BELOW ARE TO WORKSHOP

In [18]:
df['severity_score_raw']

0      6.5
1      6.5
2      0.0
3      3.0
4      0.0
5     13.5
6      8.5
7      4.0
8      3.5
9      1.0
10     6.0
11     5.0
12     5.5
13     0.0
14     0.5
15     2.5
16     2.0
17     0.0
18     3.5
19     0.0
20     0.5
21     1.0
22     1.0
23     0.0
24     1.0
25     3.0
26     0.0
27     0.0
28     1.0
29     1.0
30     0.0
31     2.0
Name: severity_score_raw, dtype: float64

In [19]:
df['severity_score_raw'].min(), df['severity_score_raw'].max()

(np.float64(0.0), np.float64(13.500000000000004))

RECALCULATION BELOW

In [24]:
severity_norm = 100 * (df['severity_score_raw'] - df['severity_score_raw'].min()) / (
    df['severity_score_raw'].max() - df['severity_score_raw'].min()
)

df['severity_score'] = 100 - severity_norm

print("Min/Max:", df['severity_score'].min(), df['severity_score'].max())

df[['Brand', 'severity_score']].sort_values(by='severity_score')

Min/Max: -1.4210854715202004e-14 100.0


,Brand,severity_score
5,Nike,-1.421085e-14
6,Under Armour,3.703704e+01
0,GAP,5.185185e+01
1,Banana Republic,5.185185e+01
10,Levi Strauss & Co.,5.555556e+01
12,Ralph Lauren Corporation,5.925926e+01
11,Wrangler,6.296296e+01
7,Lululemon,7.037037e+01
18,Converse,7.407407e+01
8,New Balance,7.407407e+01


In [25]:
df['severity_score'] = df['severity_score'].clip(lower=0)
df['severity_score'] = df['severity_score'].round(2)

df[['Brand', 'severity_score']]

,Brand,severity_score
0,GAP,51.85
1,Banana Republic,51.85
2,Old Navy,100.00
3,American Eagle,77.78
4,Abercrombie & Fitch,100.00
5,Nike,0.00
6,Under Armour,37.04
7,Lululemon,70.37
8,New Balance,74.07
9,Champion,92.59


Below are the calculations for the full HRRR metric

In [12]:
full_cols = ['HRD_Attacks', 'Other_Allegations', 'Response_Requests ', 'Response_Rate']
X_full = df[full_cols].astype(float)

X_full.head()

,HRD_Attacks,Other_Allegations,Response_Requests,Response_Rate
0,3.0,10.0,25.0,96.0
1,3.0,10.0,25.0,96.0
2,0.0,0.0,2.0,50.0
3,0.0,6.0,7.0,72.0
4,0.0,0.0,3.0,67.0


In [13]:
X_full = X_full.copy()
X_full['Response_Rate'] = -X_full['Response_Rate']

In [14]:
X_full = X_full.fillna(X_full.mean())

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(X_full)

X_full_scaled

array([[ 3.49929037,  0.88203092,  2.96832515, -1.13011182],
       [ 3.49929037,  0.88203092,  2.96832515, -1.13011182],
       [-0.40689423, -0.81826965, -0.52159068, -0.01139992],
       [-0.40689423,  0.20191069,  0.23708667, -0.54643605],
       [-0.40689423, -0.81826965, -0.36985521, -0.42483693],
       [-0.40689423,  3.77254189,  2.2096478 , -0.86259376],
       [-0.40689423,  2.07224132,  0.0853512 , -0.01139992],
       [-0.40689423,  0.54197081,  0.54055762, -1.22739112],
       [-0.40689423,  0.37194075, -0.06638427, -0.74099464],
       [-0.40689423, -0.47820953, -0.06638427,  1.20459128],
       [ 0.8951673 ,  1.05206098,  1.45097044, -0.74099464],
       [-0.40689423,  0.88203092, -0.21811974,  0.59659568],
       [ 0.8951673 ,  0.88203092,  0.38882214, -0.01139992],
       [-0.40689423, -0.81826965, -0.67332615, -1.22739112],
       [ 0.8951673 , -0.81826965, -0.67332615, -1.22739112],
       [-0.40689423,  0.03188064, -0.06638427,  0.23179832],
       [ 0.8951673 , -0.

In [16]:
from sklearn.decomposition import PCA

pca_full = PCA()
full_components = pca_full.fit_transform(X_full_scaled)

full_components

array([[ 4.47787215e+00, -6.78743822e-01,  1.61167978e+00,
         3.96829383e-02],
       [ 4.47787215e+00, -6.78743822e-01,  1.61167978e+00,
         3.96829383e-02],
       [-8.91738040e-01, -5.42039530e-01, -3.77996386e-02,
         1.29932799e-01],
       [ 2.40384141e-01, -7.22321130e-02, -6.57735609e-01,
         2.55959254e-01],
       [-6.35977770e-01, -7.89723368e-01, -2.85249204e-01,
         2.07146303e-01],
       [ 3.17081573e+00,  2.49171667e+00, -1.91265352e+00,
         3.21370086e-01],
       [ 7.67010597e-01,  1.58380320e+00, -1.03474673e+00,
        -5.47469766e-01],
       [ 8.45721834e-01, -2.32136256e-01, -1.18526219e+00,
         2.89840719e-01],
       [ 2.03902803e-01, -1.10968678e-01, -8.94344404e-01,
        -6.03360941e-02],
       [-9.34471402e-01,  5.32026284e-01,  6.97078644e-01,
         4.55507297e-01],
       [ 2.11757648e+00,  2.03962301e-01, -9.04362295e-02,
         1.69220615e-01],
       [-1.87038144e-01,  1.09220478e+00, -2.34933080e-01,
      

In [17]:
pca_full.explained_variance_ratio_

array([0.60097213, 0.20542447, 0.17175988, 0.02184352])

In [18]:
import pandas as pd

full_loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_full.columns))],
    index=X_full.columns
)

full_loadings

,PC1,PC2,PC3,PC4
HRD_Attacks,0.515837,-0.263722,0.639396,-0.505507
Other_Allegations,0.443749,0.711302,-0.376365,-0.394317
Response_Requests,0.619644,0.115031,0.149833,0.761813
Response_Rate,-0.391204,0.641302,0.653508,0.092833


In [19]:
full_weights = abs(full_loadings['PC1'])
full_weights = full_weights / full_weights.sum()

full_weights

HRD_Attacks           0.261788
Other_Allegations     0.225204
Response_Requests     0.314471
Response_Rate         0.198537
Name: PC1, dtype: float64

In [21]:
df['hr_score_raw'] = (
    full_weights['HRD_Attacks'] * df['HRD_Attacks'] +
    full_weights['Other_Allegations'] * df['Other_Allegations'] +
    full_weights['Response_Requests '] * df['Response_Requests '] +
    full_weights['Response_Rate'] * (-df['Response_Rate'])
)

In [22]:
hr_norm = 100 * (df['hr_score_raw'] - df['hr_score_raw'].min()) / (
    df['hr_score_raw'].max() - df['hr_score_raw'].min()
)

df['hr_score'] = 100 - hr_norm

df[['Brand', 'hr_score']].sort_values(by='hr_score')

,Brand,hr_score
22,Forever 21,0.000000
9,Champion,0.000000
18,Converse,2.070019
25,Patagonia,3.114466
29,Anthropologie,4.375352
28,Urban Outfitters,4.375352
31,LL Bean,5.203360
23,Tory Burch,6.464246
17,Kate Spade,7.922697
27,Tom Ford,9.381147


In [23]:
full_weights

HRD_Attacks           0.261788
Other_Allegations     0.225204
Response_Requests     0.314471
Response_Rate         0.198537
Name: PC1, dtype: float64